In [2]:
import sys
sys.path.append('..')

In [3]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)



2026-06-20 04:26:42.073475464 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [ ]:
#answer to question 1
print(f"{v1[0]:.8f}")

-0.02058203


In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [31]:
from tqdm.auto import tqdm
import numpy as np

for d in tqdm(range(0, len(documents))):
    if documents[d]["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
      print(documents[d]["filename"])
      thecontent = documents[d]["content"]
      break
thecontent   

  0%|          | 0/72 [00:00<?, ?it/s]

02-vector-search/lessons/07-sqlitesearch-vector.md


'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [33]:
thecontent_vector = embed.encode(thecontent)
thecontent_vector

array([-6.69167643e-03, -1.96193967e-02, -1.35708193e-02,  2.79713570e-02,
        6.41942652e-02,  2.62990448e-02,  8.73513725e-03,  7.13346709e-03,
       -4.91385379e-02, -1.85060009e-02, -1.84053811e-02,  5.55403037e-02,
        1.10243993e-02, -2.03783701e-02, -1.14009126e-01,  1.41751132e-02,
       -1.42544090e-02,  4.95837102e-02,  8.09291243e-02,  4.17696897e-02,
       -2.95922934e-02,  4.67177304e-03,  4.05626713e-02, -3.90501293e-02,
        2.19091567e-03,  5.23046040e-02, -1.01323557e-02, -5.98989310e-02,
        4.16823523e-02,  5.88785129e-04,  3.38646839e-02,  2.94847675e-02,
        1.86943411e-02,  1.83957273e-01, -9.16205354e-02,  4.11525594e-02,
       -5.42344414e-02, -1.79471921e-02, -9.08646905e-02, -1.80611278e-02,
       -1.64217023e-02,  8.13042594e-03, -5.24593921e-02,  6.31449688e-02,
        4.37100306e-02,  3.84262105e-02, -2.89416808e-02, -6.02209808e-02,
        7.83269571e-02, -7.00202511e-02, -1.00836881e-01, -1.33440248e-02,
       -1.71584515e-02, -

In [ ]:
#answer to question 2
dotprod = thecontent_vector.dot(v1)
print(f"{dotprod:.8f}")

0.36107027


In [ ]:
#chunking documents
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
chunks
chunks[0]["content"]

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

In [ ]:
#grab the content only from the chunks and put them in a list
texts = [chunk["content"] for chunk in chunks]

In [ ]:
#go through each of the content in batches and encode them into vectors. Put them into the X list
#then use numpy to turn X into an array
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [50]:
X.shape

(295, 384)

In [ ]:
#get the similarity scores from doing the dot product
scores = X.dot(v1)
scores.shape

(295,)

In [ ]:
#returns the index of the highest score, which will correspond to the index in chunks and texts
#and the value of the score
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.6489017718578813))

In [ ]:
#answer to question 3
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [53]:
from minsearch import VectorSearch
#minsearch will compute the dot product for you, and will also allow you to filter by keyword fields

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)


In [ ]:
#answer to question 4
q2 = "What metric do we use to evaluate a search engine?"
v2 = embed.encode(q2)
vindex.search(v2, num_results = 5)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [ ]:
#we are going to index the chunks using minsearch for comparison

from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)
index


In [60]:
q3 = "How do I store vectors in PostgreSQL?"
v3 = embed.encode(q3)

In [64]:
#running minsearch to get the results
minsearch_results = index.search(q3, num_results=5)
minsearch_results

[{'content': '# Embeddings\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we can do vector search, we need to turn our text into vectors.\nWe call this process embedding: we embed text into a vector space. The\nvectors we get back are also called "embeddings."\n\n## Word embeddings and sentence embeddings\n\nThis idea comes from\n[word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to\nplace words as points in a multi-dimensional space. Words with similar\nmeanings land close to each other.\n\nImagine a 2D space where "enroll" and "join" are near each other and\n"Docker" is far away:\n\n```text\n        · enroll\n       · join\n                   · Docker\n```\n\nThe same idea works for entire sentences:\n\n```text\nQ1: "I just discovered the course. Can I still join it?"\nQ2: "I just found out about the program. Can I still enroll?"\n\nThese two are close - they mean the same thing.\n\nQ3: "H

In [66]:
#running vector search for the same query as minsearch
vectorsearch_results = vindex.search(v3, num_results = 5)
vectorsearch_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [ ]:
#pull out the filenames and dedup the list
minsearch_filenames = list(dict.fromkeys([result["filename"] for result in minsearch_results]))
minsearch_filenames

['02-vector-search/lessons/02-embeddings.md',
 '02-vector-search/lessons/01-intro.md',
 '02-vector-search/lessons/03-embeddings-dataset.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/10-next-steps.md']

In [ ]:
#pull out the filenames and dedup the list
vectorsearch_filenames = list(dict.fromkeys([result["filename"] for result in vectorsearch_results]))
vectorsearch_filenames

['02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md']

In [ ]:

#answer to question 5
#filename only in vector search
only_in_vectorsearch_results = set(vectorsearch_filenames) - set(minsearch_filenames)
only_in_vectorsearch_results

{'02-vector-search/lessons/08-pgvector.md'}